# 2. 제한된 재검토 loop

**시나리오:** SEV1의 첫 rollback 제안이 위험하면 risk guard가 정확히 한 번만 commander에게 수정을 요청합니다.

**학습 목표:** `revision_count`, `max_revisions`, conditional loop와 종료 보장의 관계를 익힙니다.

## 중요 변수·함수

- `revision_count`: 실제 재검토 횟수입니다.
- `max_revisions`: 무한 loop를 막는 상한입니다.
- `risk_guard`: 안전하지 않으면 revise, 안전하거나 한도 소진이면 finish를 선택합니다.

In [ ]:
# 이 학습 Notebook은 외부 API/DB를 사용하지 않는 fixture 모드로 고정합니다.
import os
os.environ["APP_MODE"] = "fixture"

# Notebook 위치에서 실행해도 repository의 canonical app을 가져옵니다.
from pathlib import Path
import sys

_repo_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists()), Path.cwd())
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

# 첫 제안이 unsafe이고 두 번째 제안이 관찰 명령인 fixture를 실행합니다.
from week3.app import IncidentRequest, create_fixture_services, run_incident_response

result = run_incident_response(
    IncidentRequest(service='checkout', summary='Checkout errors after deployment', severity='SEV1'),
    create_fixture_services(),
)

In [ ]:
# loop가 한도 안에서 끝났는지 trace로 검증합니다.
assert result['revision_count'] == result['max_revisions'] == 1
assert result['agents_run'].count('commander_agent') == 2
assert result['agents_run'].count('risk_guard') == 2
{'revision_count': result['revision_count'], 'agents_run': result['agents_run']}

## 예측 과제와 해석

**예측 과제:** 두 번째 제안도 위험할 때 자동으로 세 번째 수정을 시도해야 하는지 판단하세요.

**해석:** 이 curriculum에서는 한도 소진 후 fail-closed로 사람 검토에 넘깁니다. 개선 없는 반복보다 종료 가능성이 우선입니다.